***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [第 8 章：校准](8_0_introduction.ipynb)
    * 下一节： [8.2 第一代校准（1GC）](8_2_1gc.ipynb)

***


导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=3, suppress=True)

RNG = np.random.default_rng(20260419)


校准之所以能够被写成数值问题，是因为我们可以把“观测到的受损可见度”和“由天空模型预测的理想可见度”联系起来。对单极化、方向无关的最简情形，

$$
d_{pq}(t) = g_p(t)\,m_{pq}(t)\,g_q^\ast(t) + n_{pq}(t),
$$

其中 $d_{pq}$ 是观测数据，$m_{pq}$ 是由天空模型得到的预测可见度，$g_p$ 是天线增益，$n_{pq}$ 是噪声。若把所有基线与时刻上的残差堆叠起来，那么校准就可以被看成一个非线性最小二乘问题：寻找一组参数，使残差平方和最小。

这一节不追求最一般的推导，而是把三个关键直觉讲清楚：

- 为什么最小二乘目标函数会自然出现；
- 为什么全局相位必须通过参考天线来固定；
- 为什么天空模型不完整时，求解器会把错误“吸收”进增益。


***


## 8.1 作为最小二乘问题的校准 <a id='cal:sec:cal_ls'></a>

本节围绕一个小型东西向阵列的模拟观测展开。我们会：

- 先生成一个简化的 $uv$ 覆盖；
- 用一个多点源模型构造理想可见度；
- 向数据注入时间变化的复增益；
- 用一个简化版的 Levenberg-Marquardt（LM）求解器恢复增益；
- 再观察参考天线选择和模型不完整时会发生什么。


In [ ]:
def baseline_pairs(nant):
    return [(p, q) for p in range(nant) for q in range(p + 1, nant)]


def ew_uv_tracks(ant_x_m, hour_angle_h, dec_deg, wavelength_m=0.214):
    pairs = baseline_pairs(len(ant_x_m))
    hour_angle_rad = np.deg2rad(15.0 * hour_angle_h)
    dec = np.deg2rad(dec_deg)
    u = np.zeros((hour_angle_h.size, len(pairs)))
    v = np.zeros_like(u)

    for ti, ha in enumerate(hour_angle_rad):
        for bi, (p, q) in enumerate(pairs):
            baseline_lambda = (ant_x_m[q] - ant_x_m[p]) / wavelength_m
            u[ti, bi] = baseline_lambda * np.sin(ha)
            v[ti, bi] = -baseline_lambda * np.sin(dec) * np.cos(ha)

    return pairs, u, v


def sky_model_vis(u, fluxes, ls):
    vis = np.zeros_like(u, dtype=complex)
    for flux, ll in zip(fluxes, ls):
        vis += flux * np.exp(-2j * np.pi * u * ll)
    return vis


def gain_track(times_h, nant):
    ant_phase = np.linspace(0.0, np.pi, nant, endpoint=False)
    amp = 1.0 + 0.06 * np.sin(0.9 * times_h[:, None] + 0.6 * ant_phase[None, :])
    phase = 0.45 * np.sin(1.7 * times_h[:, None] + ant_phase[None, :])
    return amp * np.exp(1j * phase)


def apply_gains(model_vis, gains, pairs, noise_std=0.0, rng=None):
    data = np.zeros_like(model_vis, dtype=complex)
    for bi, (p, q) in enumerate(pairs):
        data[:, bi] = gains[:, p] * model_vis[:, bi] * np.conj(gains[:, q])

    if noise_std > 0.0:
        if rng is None:
            rng = np.random.default_rng(0)
        data += noise_std * (
            rng.normal(size=data.shape) + 1j * rng.normal(size=data.shape)
        )

    return data


def pack_params(gains, ref_ant=0):
    log_amp = np.log(np.maximum(np.abs(gains), 1e-12))
    phase = np.angle(gains)
    params = list(log_amp)
    for ant in range(len(gains)):
        if ant == ref_ant:
            continue
        params.append(phase[ant] - phase[ref_ant])
    return np.array(params, dtype=float)


def unpack_params(params, nant, ref_ant=0):
    log_amp = np.array(params[:nant])
    phase = np.zeros(nant)
    idx = nant
    for ant in range(nant):
        if ant == ref_ant:
            continue
        phase[ant] = params[idx]
        idx += 1
    return np.exp(log_amp + 1j * phase)


def predict_vis(model_vec, pairs, gains):
    pred = np.zeros_like(model_vec, dtype=complex)
    for bi, (p, q) in enumerate(pairs):
        pred[bi] = gains[p] * model_vec[bi] * np.conj(gains[q])
    return pred


def residual_vector(params, model_vec, data_vec, pairs, nant, ref_ant=0):
    gains = unpack_params(params, nant, ref_ant=ref_ant)
    resid = data_vec - predict_vis(model_vec, pairs, gains)
    return np.concatenate([resid.real, resid.imag])


def numerical_jacobian(fun, params, eps=1e-6):
    base = fun(params)
    jac = np.zeros((base.size, params.size))
    for idx in range(params.size):
        step = eps * max(1.0, abs(params[idx]))
        shifted = params.copy()
        shifted[idx] += step
        jac[:, idx] = (fun(shifted) - base) / step
    return jac, base


def lm_solve(data_vec, model_vec, pairs, nant, ref_ant=0, n_iter=12, lambda0=1e-2):
    params = pack_params(np.ones(nant, dtype=complex), ref_ant=ref_ant)
    lam = lambda0
    history = []

    def fun(x):
        return residual_vector(x, model_vec, data_vec, pairs, nant, ref_ant=ref_ant)

    for _ in range(n_iter):
        jac, resid = numerical_jacobian(fun, params)
        sse = float(resid @ resid)
        history.append(np.sqrt(np.mean(resid**2)))

        jtj = jac.T @ jac
        grad = jac.T @ resid
        damping = np.diag(np.diag(jtj)) + 1e-9 * np.eye(jtj.shape[0])
        delta = np.linalg.solve(jtj + lam * damping, -grad)

        trial = params + delta
        resid_trial = fun(trial)
        if float(resid_trial @ resid_trial) < sse:
            params = trial
            lam *= 0.7
        else:
            lam *= 2.5

    history.append(np.sqrt(np.mean(fun(params) ** 2)))
    return unpack_params(params, nant, ref_ant=ref_ant), np.array(history)


def solve_track(data, model, pairs, nant, ref_ant=0, n_iter=12):
    gains = np.zeros((data.shape[0], nant), dtype=complex)
    histories = []
    for t in range(data.shape[0]):
        gains[t], hist = lm_solve(
            data[t],
            model[t],
            pairs,
            nant,
            ref_ant=ref_ant,
            n_iter=n_iter,
        )
        histories.append(hist)
    return gains, np.array(histories)


def correct_vis(data, gains, pairs):
    corrected = np.zeros_like(data, dtype=complex)
    for bi, (p, q) in enumerate(pairs):
        corrected[:, bi] = data[:, bi] / (
            gains[:, p] * np.conj(gains[:, q]) + 1e-12
        )
    return corrected


def rms_complex(arr):
    return np.sqrt(np.mean(np.abs(arr) ** 2))


### 8.1.1 先生成一个简化的 $uv$ 覆盖 <a id='cal:sec:uv'></a>

为了让重点落在求解器上，这里只考虑一个东西向阵列，并把天体源限制在 $m=0$ 的一维方向余弦轴上。即便如此，随着地球自转，阵列仍会在 $uv$ 平面上扫出一组有代表性的轨迹。


In [ ]:
nant = 5
ant_x_m = np.array([0.0, 36.0, 102.0, 210.0, 348.0])
times_h = np.linspace(-3.5, 3.5, 16)
dec_deg = 50.0

pairs, u, v = ew_uv_tracks(ant_x_m, times_h, dec_deg=dec_deg, wavelength_m=0.214)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].scatter(ant_x_m, np.zeros_like(ant_x_m), s=80, color="tab:blue")
for idx, x in enumerate(ant_x_m):
    axes[0].text(x + 4.0, 0.02, f"A{idx}", fontsize=10)
axes[0].set_xlabel("east-west position [m]")
axes[0].set_ylabel("north-south position [m]")
axes[0].set_title("Simplified east-west array")

for bi in range(u.shape[1]):
    axes[1].plot(u[:, bi], v[:, bi], lw=1.3)
    axes[1].plot(-u[:, bi], -v[:, bi], lw=1.0, alpha=0.45)
axes[1].set_xlabel("u [wavelength]")
axes[1].set_ylabel("v [wavelength]")
axes[1].set_title("Earth rotation synthesis tracks")

plt.tight_layout()
print(f"天线数 = {nant}，基线数 = {len(pairs)}")


在这个几何设置上，我们用三个点源来构造天空模型。由于源都取在 $m=0$ 上，模型可见度只显式依赖于 $u$：

$$
m_{pq}(u) = \sum_s I_s \exp(-2\pi i u l_s).
$$

这种简化并不会改变校准的核心数学结构，但它能让我们更容易看清参数退化和求解行为。


### 8.1.2 最小二乘目标函数与参考天线 <a id='cal:sec:RIME_un'></a>

令参数向量 $\boldsymbol{\theta}$ 包含各天线的对数振幅和相位，则最小二乘目标函数可以写成

$$
\chi^2(\boldsymbol{\theta}) =
\sum_{t,p<q}
\left|d_{pq}(t) - g_p(\boldsymbol{\theta}, t)\,
m_{pq}(t)\,
g_q^\ast(\boldsymbol{\theta}, t)\right|^2.
$$

对该式有一个非常关键的观察：若把所有增益同时乘上同一个相位因子 $e^{i\phi_0}$，那么每条基线上的 $g_p g_q^\ast$ 都不会改变。因此，$\chi^2$ 对全局相位并不敏感，这就是为什么必须通过参考天线把这一自由度固定下来。


In [ ]:
fluxes_true = np.array([1.0, 0.35, 0.18])
ls_true = np.array([0.0, 0.012, -0.021])
model_vis_true = sky_model_vis(u, fluxes_true, ls_true)

true_gains = gain_track(times_h, nant)
data_vis = apply_gains(model_vis_true, true_gains, pairs, noise_std=0.02, rng=RNG)

t0 = 8
global_phases = np.linspace(-np.pi, np.pi, 200)
chi2_global = []
for phase in global_phases:
    shifted = true_gains[t0] * np.exp(1j * phase)
    resid = data_vis[t0] - predict_vis(model_vis_true[t0], pairs, shifted)
    chi2_global.append(np.sum(np.abs(resid) ** 2))

fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(global_phases, chi2_global, color="tab:purple", lw=2.0)
ax.set_xlabel("applied global phase shift [rad]")
ax.set_ylabel("chi-square")
ax.set_title("A global phase shift leaves the least-squares cost unchanged")
plt.tight_layout()
print(f"chi-square variation = {max(chi2_global) - min(chi2_global):.3e}")


这条近乎水平的曲线表明：如果不固定参考相位，求解器就会在一个平坦方向上游走。实际处理里，常见做法是把一面信号稳定、性能可靠的天线选作参考天线，只固定它的相位而不强行固定绝对振幅。


### 8.1.3 生成受污染可见度与初始残差 <a id='cal:sec:sim'></a>

下面把时间变化的复增益施加到理想可见度上，并加入热噪声。这样我们就得到了一个最小但完整的校准输入：模型 $m_{pq}(t)$、观测数据 $d_{pq}(t)$，以及一组待估参数 $g_p(t)$。


In [ ]:
baseline = (1, 4)
baseline_idx = pairs.index(baseline)
raw_residual_rms = rms_complex(data_vis - model_vis_true)

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex="col")

axes[0, 0].plot(times_h, np.abs(true_gains[:, 3]), color="tab:blue", lw=2.0)
axes[0, 0].set_ylabel("|g_3|")
axes[0, 0].set_title("True amplitude track of antenna 3")

axes[1, 0].plot(times_h, np.angle(true_gains[:, 3]), color="tab:orange", lw=2.0)
axes[1, 0].set_ylabel("phase [rad]")
axes[1, 0].set_xlabel("hour angle [hour]")
axes[1, 0].set_title("True phase track of antenna 3")

axes[0, 1].plot(times_h, np.abs(model_vis_true[:, baseline_idx]), color="black", lw=2.0, label="model")
axes[0, 1].plot(times_h, np.abs(data_vis[:, baseline_idx]), color="tab:red", lw=1.6, label="data")
axes[0, 1].set_ylabel("amplitude [Jy]")
axes[0, 1].set_title(f"Baseline {baseline[0]}-{baseline[1]} amplitude")
axes[0, 1].legend(loc="upper right")

axes[1, 1].plot(
    times_h,
    np.angle(data_vis[:, baseline_idx] / (model_vis_true[:, baseline_idx] + 1e-12)),
    color="tab:red",
    lw=1.6,
)
axes[1, 1].axhline(0.0, color="black", ls="--")
axes[1, 1].set_ylabel("phase offset [rad]")
axes[1, 1].set_xlabel("hour angle [hour]")
axes[1, 1].set_title(f"Baseline {baseline[0]}-{baseline[1]} phase offset")

plt.tight_layout()
print(f"未校准时，对真模型的可见度 RMS 残差 = {raw_residual_rms:.4f}")


从这个时刻起，问题就完全变成了数值优化：我们要找到一组参数，使模型在所有基线上的联合误差最小。由于 $g_p m_{pq} g_q^\ast$ 对参数是非线性的，所以通常需要迭代求解，而不是一次线性代数运算就结束。


### 8.1.4 用一个简化的 LM 求解器恢复增益 <a id='cal:sec:LM'></a>

这里我们实现一个小型的 LM 风格求解器。它的核心思想是：在当前参数点附近，用雅可比矩阵把残差局部线性化，然后通过一个带阻尼的正规方程给出参数更新。阻尼项较大时，它更像稳定但保守的梯度下降；阻尼项较小时，它更接近标准高斯-牛顿步骤。


In [ ]:
solved_gains_ref0, histories_ref0 = solve_track(
    data_vis,
    model_vis_true,
    pairs,
    nant,
    ref_ant=0,
    n_iter=12,
)
corrected_ref0 = correct_vis(data_vis, solved_gains_ref0, pairs)

true_gains_ref0 = true_gains * np.exp(-1j * np.angle(true_gains[:, [0]]))
solved_residual_rms = rms_complex(corrected_ref0 - model_vis_true)
gain_amp_rms = np.sqrt(
    np.mean((np.abs(solved_gains_ref0[:, 3]) - np.abs(true_gains_ref0[:, 3])) ** 2)
)
gain_phase_rms = np.sqrt(
    np.mean((np.angle(solved_gains_ref0[:, 3] / true_gains_ref0[:, 3])) ** 2)
)

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex="col")

axes[0, 0].plot(histories_ref0.mean(axis=0), marker="o", color="tab:purple", lw=1.8)
axes[0, 0].set_ylabel("RMS residual")
axes[0, 0].set_title("Mean LM convergence history")

axes[1, 0].plot(times_h, np.abs(true_gains_ref0[:, 3]), color="black", lw=2.0, label="true")
axes[1, 0].plot(times_h, np.abs(solved_gains_ref0[:, 3]), color="tab:blue", lw=1.7, label="solved")
axes[1, 0].set_ylabel("|g_3|")
axes[1, 0].set_xlabel("hour angle [hour]")
axes[1, 0].legend(loc="upper right")

axes[0, 1].plot(times_h, np.angle(true_gains_ref0[:, 3]), color="black", lw=2.0, label="true")
axes[0, 1].plot(times_h, np.angle(solved_gains_ref0[:, 3]), color="tab:orange", lw=1.7, label="solved")
axes[0, 1].set_ylabel("phase [rad]")
axes[0, 1].set_title("Antenna 3 phase relative to reference antenna")
axes[0, 1].legend(loc="upper right")

axes[1, 1].plot(times_h, np.abs(data_vis[:, baseline_idx]), color="tab:red", lw=1.5, label="raw data")
axes[1, 1].plot(times_h, np.abs(corrected_ref0[:, baseline_idx]), color="tab:blue", lw=1.8, label="corrected")
axes[1, 1].plot(times_h, np.abs(model_vis_true[:, baseline_idx]), color="black", lw=2.0, alpha=0.7, label="model")
axes[1, 1].set_ylabel("amplitude [Jy]")
axes[1, 1].set_xlabel("hour angle [hour]")
axes[1, 1].set_title(f"Corrected visibility on baseline {baseline[0]}-{baseline[1]}")
axes[1, 1].legend(loc="upper right")

plt.tight_layout()
print(f"校准后，对真模型的可见度 RMS 残差 = {solved_residual_rms:.4f}")
print(f"天线 3 振幅 RMS 误差 = {gain_amp_rms:.4f}")
print(f"天线 3 相位 RMS 误差 = {gain_phase_rms:.4f} rad")


这里最值得注意的是两件事：

- 残差在很少几步内就快速下降，说明局部线性化在这个问题上是有效的；
- 我们比较的是“相对于参考天线相位归一化后的真实增益”和求解结果，而不是原始绝对相位轨迹。

这再次提醒我们：校准解本身带有规范选择，不能脱离参考定义来解读。


### 8.1.5 更换参考天线后会发生什么 <a id='cal:sec:cor'></a>

下面把参考天线从 0 号改成 2 号。由于我们固定的是全局相位规范，而不是物理信号本身，因此**增益参数的表达会改变，但改正后的数据应该保持不变**。


In [ ]:
solved_gains_ref2, histories_ref2 = solve_track(
    data_vis,
    model_vis_true,
    pairs,
    nant,
    ref_ant=2,
    n_iter=12,
)
corrected_ref2 = correct_vis(data_vis, solved_gains_ref2, pairs)
correction_difference = rms_complex(corrected_ref0 - corrected_ref2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(times_h, np.angle(solved_gains_ref0[:, 3]), color="tab:blue", lw=1.8, label="ref = antenna 0")
axes[0].plot(times_h, np.angle(solved_gains_ref2[:, 3]), color="tab:green", lw=1.8, label="ref = antenna 2")
axes[0].set_xlabel("hour angle [hour]")
axes[0].set_ylabel("phase [rad]")
axes[0].set_title("Gain phases depend on the chosen reference")
axes[0].legend(loc="upper right")

axes[1].plot(
    times_h,
    np.abs(corrected_ref0[:, baseline_idx] - corrected_ref2[:, baseline_idx]),
    color="tab:red",
    lw=1.8,
)
axes[1].set_xlabel("hour angle [hour]")
axes[1].set_ylabel("|difference| [Jy]")
axes[1].set_title("Corrected visibilities remain invariant")
plt.tight_layout()

print(f"两种参考天线下，改正后可见度的 RMS 差异 = {correction_difference:.4e}")


因此，参考天线的选择影响的是“解的表示方式”，而不是最终应该得到的物理校正结果。真正需要小心的是：若参考天线本身不稳定，那么它会把额外噪声传播到整组相位解里。


### 8.1.6 当天空模型不完整时，求解器会吸收错误

最后做一个在真实处理里非常重要的实验：故意把第三个较弱源从模型里删掉，再重复一次求解。这样做之后，求解器仍会努力减小残差，但它面对的是一个“先天错误的问题”。这时部分天空结构会被错当成增益变化。


In [ ]:
fluxes_incomplete = np.array([1.0, 0.35])
ls_incomplete = np.array([0.0, 0.012])
model_vis_incomplete = sky_model_vis(u, fluxes_incomplete, ls_incomplete)

solved_gains_bad, histories_bad = solve_track(
    data_vis,
    model_vis_incomplete,
    pairs,
    nant,
    ref_ant=0,
    n_iter=12,
)
corrected_bad = correct_vis(data_vis, solved_gains_bad, pairs)

residual_good = rms_complex(corrected_ref0 - model_vis_true)
residual_bad_true = rms_complex(corrected_bad - model_vis_true)
residual_bad_model = rms_complex(corrected_bad - model_vis_incomplete)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(histories_ref0.mean(axis=0), marker="o", lw=1.8, color="tab:blue", label="complete model")
axes[0].plot(histories_bad.mean(axis=0), marker="s", lw=1.8, color="tab:red", label="incomplete model")
axes[0].set_xlabel("LM iteration")
axes[0].set_ylabel("mean RMS residual")
axes[0].set_title("A wrong sky model stalls the solver")
axes[0].legend(loc="upper right")

axes[1].bar(
    ["complete->true", "incomplete->true", "incomplete->its model"],
    [residual_good, residual_bad_true, residual_bad_model],
    color=["tab:blue", "tab:red", "tab:orange"],
)
axes[1].set_ylabel("visibility RMS residual")
axes[1].set_title("Model incompleteness leaves structured residuals")
plt.tight_layout()

print(f"完整模型校准后，对真模型的 RMS 残差 = {residual_good:.4f}")
print(f"不完整模型校准后，对真模型的 RMS 残差 = {residual_bad_true:.4f}")
print(f"不完整模型校准后，对其自身模型的 RMS 残差 = {residual_bad_model:.4f}")


这正是最小二乘校准最容易被误用的地方。求解器只会最小化你给它的目标函数，而不会替你判断天空模型是否正确。也就是说：

- **残差下降**不等于**物理解释正确**；
- **解看起来平滑**不等于**没有把天空结构吃进增益**；
- 一旦发现残差无法下降到接近噪声，首先要怀疑模型，而不只是继续调求解器参数。

完成本节后，再去读 [8.2 第一代校准（1GC）](8_2_1gc.ipynb) 会更容易理解：为什么校准源的已知模型如此重要，以及为什么 1GC 的成功往往建立在“模型足够简单、误差近似方向无关”之上。
